### Line Critical Infrastructure Data to Hexbin

In [ ]:
import os

os.chdir("C")

os.getcwd()

#summarizing the linear infrastructure data into hex bins
#summarizing by mileage and making and index for each

#The best CRS is dependent on your project area for mileage calculations.
#Calculations may be different based on which CRS you need.

In [3]:
import numpy as np
from shapely.geometry import Point, Polygon
import pandas as pd
import matplotlib.pyplot as plt
import folium
import base64
import geopandas as gpd
import pathlib as Path
import geopandas as gpd
from shapely import union_all
import fiona
import os


In [ ]:
#reading the buffered hex bins that was already made -
#the buffered hex bins was a geopackage
hex_path = ""
#Define for GPD:
gdf_hex = gpd.read_file(hex_path)


#Read all the Line Data :
#All line infrastructure data should be in one folder together
folder = r""

gdfs = []  # initialize list

for file in os.listdir(folder):
    if file.endswith(".shp"):
        full_path = os.path.join(folder, file)
        gdf = gpd.read_file(full_path)
        gdfs.append(gdf)
# Ensure same crs
gdfs = [gdf.to_crs("EPSG:4326") for gdf in gdfs]
# Combine into one GeoDataFrame
combined_gdf = gpd.GeoDataFrame(pd.concat(gdfs, ignore_index=True))
# Ensure same crs
combined_gdf = [combined_gdf.to_crs("EPSG:4326")]
# Ensure it is a geodataframe correctly - was a list before
combined_gdf = gpd.GeoDataFrame(
    pd.concat(gdfs, ignore_index=True),
    crs=gdfs[0].crs
)

print(combined_gdf.crs)
print(combined_gdf.head())

In [ ]:
#Checking for geodataframe and not list
type(combined_gdf)

In [ ]:
# Check tthe crs
print(gdf_hex.crs), print(combined_gdf.crs)

In [ ]:
#ESPG:4326 is in decimal degrees. We will need to change, temporarily project for lenghth
#The best CRS is dependent on your project area
projected_crs = "EPSG:3968"  # NAD83 / Virginia Lambert

# Project ONLY for intersection + length calculation
lines_proj = combined_gdf.to_crs(projected_crs)
hex_proj = gdf_hex.to_crs(projected_crs)

# Intersect lines with hex polygons (only once)
intersections = gpd.overlay(
    lines_proj,
    hex_proj,
    how="intersection",
    keep_geom_type=True
)

# Calculate length in meters → convert to miles
intersections["miles"] = (
    intersections.geometry.length * 0.000621371
)


#Summarize by SubSector
subsector_miles = (
    intersections
    .groupby(["h3_ID", "Sub-Sector"])["miles"]
    .sum()
    .reset_index()
)

subsector_wide = (
    subsector_miles
    .pivot(index="h3_ID", columns="Sub-Sector", values="miles")
    .fillna(0)
    .reset_index()
)

# Merge back to hex bin

hex_summary = gdf_hex.merge(
    subsector_wide,
    on="h3_ID",
    how="left"
)

# Replace NaNs with 0
hex_summary = hex_summary.fillna(0)


In [ ]:
#Check if intersections actually worked
print(len(intersections))
print(intersections.head())

In [ ]:
#checking Groupby results
print(subsector_miles.head())

In [10]:
#Let's go ahead and try to fix the dtype thing
for col in hex_summary.select_dtypes(include=["float64"]).columns:
    
    values = hex_summary[col].dropna()
    
    if len(values) > 0 and np.all(np.isclose(values % 1, 0)):
        hex_summary[col] = hex_summary[col].round().astype("Int64")
        
for col in hex_summary.select_dtypes(include=["float64"]).columns:
    
    # Skip geometry if present
    if col == "geometry":
        continue
    
    # Check if values are effectively integers (handles float precision)
    if np.all(np.isclose(hex_summary[col] % 1, 0)):
        hex_summary[col] = hex_summary[col].round().astype("Int64")

In [ ]:
#Let's Check
print(hex_summary.dtypes)

In [ ]:
#let's check 
total = hex_summary['Bridges'].sum()
print(total)

In [ ]:
hex_summary.head()

In [ ]:
#Can map and check numbers inside of hex bins
#hex_summary.explore()

### Transportation Group

In [ ]:
trans_cols = [
    "Aviation Runways",
    "Bridges",
    "Bus Network",
    "Culverts",
    "Footbridges",
    "Hazardous Materials Routes",
    "Hurricane Evacuation Route",
    "Rail Lines",
    "Strategic Highway Network"
]

# Ensure numeric (invalid values → NaN)
hex_summary[trans_cols] = hex_summary[trans_cols].apply(pd.to_numeric, errors="coerce")

# Create total column (skip NaNs automatically)
hex_summary["Transportation_Miles"] = (
    hex_summary[trans_cols]
    .sum(axis=1, min_count=1)   # keeps NaN if all inputs are NaN
    .fillna(0)                  # replace all-NaN rows with 0
    .round()
    .astype("Int64")            # nullable integer
)

# Min-max normalization (0–1)
min_val = hex_summary["Transportation_Miles"].min()
max_val = hex_summary["Transportation_Miles"].max()

if min_val == max_val:
    hex_summary["T_Mi_Index"] = 0
else:
    hex_summary["T_Mi_Index"] = (
        (hex_summary["Transportation_Miles"] - min_val) / (max_val - min_val)
    )

print(hex_summary["T_Mi_Index"].describe())

### Energy Group

In [ ]:
energy_cols = [
    "Electric Transmission Lines",
    "Natural Gas Pipelines"
]

# Ensure numeric (invalid values → NaN)
hex_summary[energy_cols] = hex_summary[energy_cols].apply(pd.to_numeric, errors="coerce")

# Create total column (skip NaNs automatically)
hex_summary["Energy_Miles"] = (
    hex_summary[energy_cols]
    .sum(axis=1, min_count=1)   # keeps NaN if all inputs are NaN
    .fillna(0)                  # replace all-NaN rows with 0
    .round()
    .astype("Int64")            # nullable integer
)

# Min-max normalization (0–1)
min_val = hex_summary["Energy_Miles"].min()
max_val = hex_summary["Energy_Miles"].max()

if min_val == max_val:
    hex_summary["E_Mi_Index"] = 0
else:
    hex_summary["E_Mi_Index"] = (
        (hex_summary["Energy_Miles"] - min_val) / (max_val - min_val)
    )

print(hex_summary["E_Mi_Index"].describe())

### Save

In [ ]:
#Save to file
#Keep a geopackage here - want LONG column names to be maintained for the time being.
#name and save where H3 Edits are going 
hex_summary.to_file(r'.gpkg',
    driver="GPKG")